# 4-3: Geospatial Data in Python

In this tutorial, we'll learn how to create maps with Python. We'll start big picture: what is geospatial data? How do Python libraries like GeoPandas store geospatial data? How do libraries like Shapely represent geometry? We'll address these things by looking at national and state boundary maps. Then we'll look at creating maps using latitude and longitude data from a newspaper dataset ([the Chronicling America archive](https://guides.loc.gov/chronicling-america/)).

As you move through the notebook, pay attention to the division of labor among the libraries. `pandas` handles ordinary tables, `GeoPandas` handles tables with map geometries, `Shapely` creates geometry objects such as points, `matplotlib` supports the static plots, and `folium` builds the interactive web map.

## Learning Objectives

1. Explain what makes geospatial data different from an ordinary table
2. Load national and state boundary files with GeoPandas
3. Create point geometries from latitude and longitude columns with Shapely
4. Map historical newspaper data from `chron_am_papers.csv`
5. Jitter repeated locations so overlapping points become easier to see
6. Build Folium hover labels and popups with newspaper names, links, and location information
7. Scale map points with `Number of Issues`

## Import the Libraries

We'll use several libraries together:

- `pandas` as always
- `geopandas` for spatial tabular data
- `shapely` for geometry objects like points and polygons
- `folium` for interactive web maps
- `matplotlib` as always for our little data viz elements
- `numpy` for numeric transformations and random jitter
- `requests` for downloading boundary files from the web
- `pathlib.Path` for working with file and folder paths
- `html.escape` for making safe popup text in the interactive map

This whole pipeline requires some synchronicity between libraries. Each has its role that we'll describe in detail, but the shortest version is: 

1) we use `pandas` to clean and reshape tabular data
2) we use `GeoPandas` to read, store, filter, and plot spatial tables
3) we use `Shapely` to create point geometries that go inside a GeoDataFrame
4) we use `numpy` to help us calculate slight coordinate changes and scaled marker sizes
5) and finally, we use `folium` to turn the final data into an interactive browser map

Let's get to it:

In [ ]:
from pathlib import Path
from itertools import combinations
from html import escape

import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import matplotlib.pyplot as plt
import requests

from shapely.geometry import Point

## What Is Geospatial Data?

But first, what is geospatial data?

![geo 1](../img/Geo1.png)

Geospatial data is any data that has location information attached to it. Sometimes that location is a point, such as a city or town. Sometimes it's a line, such as a road or river. Sometimes it's an area, such as a state, county, country, neighborhood, or building footprint.

There are several Python libraries designed to interact with and understand geospatial data. Among them, `geopandas` is popular because, like `pandas`, it handles data in tabular form. But there are differences. In ordinary `pandas`, a row might describe a newspaper. In `geopandas`, that same row can also include a special `geometry` column that tells Python where the newspaper should appear on a map.

In this notebook, `GeoPandas` is the main library for spatial tables. The shapes inside its `geometry` column are geometry objects, which are usually created or represented with `Shapely`.

## Raster Data vs. Vector Data

In this notebook, we're working with __vector data__: data that stores geography as shapes made from coordinates (points, lines, and polygons). This is a common format for geospatial data, one that makes it easy to transfer the data into maps.

![vector data](../img/vector_data.png)

This isn't the only way to work with geospatial data, though. Sometimes, you'll find geospatial data in the form of __raster data__: data that stores geography as a grid of cells or pixels. With raster data, each cell has a value––things like elevation, temperature, satellite imagery color. This changes how you can interact with the data.

![raster data](../img/raster_data.png)

If you're trying to work with geospatial data and finding that it's incompatible with your processes, it may be due to a mismatch between your Python methods and the raster vs. vector data differences. We're working with vector data today, but if you find projects involving raster data, try looking up the `Rasterio` library. If you want to understand the differences in greater detail, you can also [check out this overview of raster and vector data approaches in Python](https://www.geoapify.com/python-geospatial-data-analysis/).

## Three Common Geometry Types

Geospatial vector data usually uses three kinds of shapes:

- __Point__: one x/y location, such as a newspaper city
- __LineString__: a connected sequence of points, such as a street
- __Polygon__: an enclosed area, such as a state boundary

Today we'll mostly use polygons for base maps and points for newspapers. `Shapely` is the library we'll use to represent these geometry types. `GeoPandas` is the library that stores many Shapely geometries together in a table.

## Latitude and Longitude

Most web maps use latitude and longitude.

- __Latitude__ tells us north/south position
- __Longitude__ tells us east/west position

Shapely makes it easy to interact with these data. There's a Shapely function called `Point()` that creates lat/long points from float values.

One tricky thing, though: Shapely points are created as `Point(longitude, latitude)`, not `Point(latitude, longitude)`. In my brain, latitude always comes first for some reason, but not with Shapely. Instead, Shapely designed its function to think of longitude as the x-coordinate (east/west runs horizontal) and latitude as the y-coordinate (north/south runs vertical). So the function thinks of these things as `Point(x, y)`.

Here's an example. Let's make a point:

In [ ]:
# Berkeley, California
# Longitude comes first, latitude comes second
berkeley_point = Point(-122.2727, 37.8715)

berkeley_point

You can always look up longitude and latitude points online to double-check. [Here's a free site to do so.](https://www.gps-coordinates.net/) But of course, a point with no other context doesn't tell you much. The `berkeley_point` object is more to show how you can interact with any Shapely points:

In [ ]:
type(berkeley_point)

In [ ]:
# A Shapely Point stores its x and y values
print("x - longitude:", berkeley_point.x)
print("y - latitude:", berkeley_point.y)

## Coordinate Reference Systems

A Coordinate Reference System, often shortened to **CRS**, explains how coordinates should be interpreted. The latitude/longitude system used by GPS and many web maps is commonly called `EPSG:4326`.

Whenever we create a GeoDataFrame from latitude and longitude, we should tell GeoPandas that the CRS is `EPSG:4326`. `GeoPandas` stores and checks the CRS for a whole GeoDataFrame. A standalone `Shapely` point only stores coordinate numbers. By itself, it doesn't know what CRS those numbers use, but when part of a GeoDataFrame, we can ensure these points get read using the proper GPS system.

## Download Boundary Data

We'll use two public boundary datasets that you can download directly:

- [Natural Earth](https://www.naturalearthdata.com/) country boundaries for a national/world map.
- [U.S. Census](https://www2.census.gov/) cartographic state boundaries for a state map.

To download these data, we'll use `requests` and `pathlib`. If you want to review them by hand, though, you can download them here:

- [Natural Earth country boundaries](https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip)
- [U.S. Census state boundaries](https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_state_20m.zip)

You'll notice they are `.zip` files. When unpacked, they contain several different file types. These packages of data can be hard to read by hand, but basically, they store one map dataset across several companion files: the `.shp` file contains the boundary shapes, the `.dbf` file contains the table of information about each shape, the `.prj` file describes the coordinate system, and the other files provide indexing, encoding, and metadata. GeoPandas knows how to read these pieces together and turn them into one GeoDataFrame.

First lets make a pathway to save our data using `Path()` and the `.mkdir` method:

In [ ]:
map_data_folder = Path("../data/geospatial")
map_data_folder.mkdir(parents=True, exist_ok=True)

Then we'll assign the urls and filenames to objects alongside the pathways:

In [ ]:
countries_url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
states_url = "https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_state_20m.zip"

countries_zip = map_data_folder / "ne_110m_admin_0_countries.zip"
states_zip = map_data_folder / "cb_2024_us_state_20m.zip" 

And here's a function for putting it all together. It also checks whether we already downloaded the data and skips if so:

In [ ]:
def download_file(url, file_path):
    """Download a file once and save it for later notebook runs."""
    if file_path.exists():
        print(f"Already downloaded: {file_path.name}")
    else:
        print(f"Downloading: {file_path.name}")
        response = requests.get(url)
        response.raise_for_status()
        file_path.write_bytes(response.content)


download_file(countries_url, countries_zip)
download_file(states_url, states_zip)

## Read a National Boundary Map

All right, now let's see what our data holds. We can do this with GeoPandas. `gpd.read_file()` is the GeoPandas version of `pd.read_csv()`, but it reads spatial data formats such as shapefiles, GeoJSON files, and zipped shapefiles.

The result is a `GeoDataFrame`: a table with a special geometry column. Notice how we also use the `.to_crs()` method to ensure the GeoDataFrame is read with the proper coordinate system:

In [ ]:
countries = gpd.read_file(countries_zip).to_crs("EPSG:4326")

countries.head()

Looks similar to a Pandas DatafRame, yes? But it's technically a different data structure:

In [ ]:
type(countries)

Still, it has a similar structure to a typical dataframe:

In [ ]:
# How many rows and columns are in this map dataset?
countries.shape

But what sets it apart is that `geometry` column:

In [ ]:
# Look at the first few rows in geometry
countries["geometry"].head()

We have multipolygons and polygons followed by coordinates. These provide specific encodings designed to work with GeoPandas. They are their own data structure:

In [ ]:
# The geometry column contains a new data structure: `geometry` data
countries.dtypes.tail()

Let's look at a single example:

In [ ]:
print(countries[["SOVEREIGNT", "geometry"]].iloc[0])
print(countries["geometry"][0])

And finally, to inspect this GeoDataFrame's encoding for its coordinate reference system, we can do this:

In [ ]:
# Check the crs
countries.crs

## Plot Country Boundaries

GeoPandas makes plotting easy with its `.plot()` method. This allows use to make a quick map with one line of code, showing what the GeoDataFrame looks like all together.

We also have options to control the colors of the map. We have the fill color (`color=`), boundary color (`edgecolor=`), boundary width (`linewidth=`), and figure size (good ole `figsize=`). For more options, check out the [GeoPandas documentation](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.plot.html).

Another bit worth noting: the plotting call starts with `GeoPandas`, but it uses `matplotlib` behind the scenes. That's why the result is stored as `ax`, a matplotlib axes object, and why we can call methods like `ax.set_title()` and `ax.set_axis_off()`.

In [ ]:
# plot automatically knows to look at the geometry column
ax = countries.plot(
    color="white",
    edgecolor="black",
    linewidth=0.4,
    figsize=(14, 8)
)

ax.set_title("World Country Boundaries")
ax.set_axis_off()

## Select the United States

A GeoDataFrame is still a table, so we can filter rows the same way we filter a pandas DataFrame. Here we keep the row where the `ADMIN` column (country or territory name column) is `United States of America`:

In [ ]:
usa = countries.loc[countries["ADMIN"] == "United States of America"]

usa[["ADMIN", "CONTINENT", "geometry"]]

Notice how this usa object is still a GeoDataFrame?

In [ ]:
type(usa)

That means we can use `.plot()` to create a map of it, too:

In [ ]:
ax = usa.plot(
    color="lightgreen",
    edgecolor="darkgreen",
    linewidth=0.8,
    figsize=(10, 6)
)

ax.set_title("United States Boundary from Natural Earth")
ax.set_axis_off()

## Read State Boundaries

Let's load U.S. state boundaries from the Census cartographic boundary files:

Library note: this section uses `GeoPandas` again. `gpd.read_file()` reads the zipped shapefile, and `.to_crs("EPSG:4326")` puts the state geometries into latitude/longitude coordinates.

In [ ]:
states = gpd.read_file(states_zip).to_crs("EPSG:4326")

states.shape

And inspect this geodataframe. Notice it also has that definitive `geometry` column:

In [ ]:
states.head()

And double-check our coordinate system:

In [ ]:
states.crs

## Plot State Boundaries

And just as before, we can easily plot this GeoDataFrame:

In [ ]:
ax = states.plot(
    facecolor="white",
    edgecolor="black",
    linewidth=0.5,
    figsize=(14, 8)
)

ax.set_title("U.S. State and Territory Boundaries")
ax.set_axis_off()

## Filter to a Few States

And just like with the Natural Earth data, we can subset this by specific states. If you look closely at the columns, you'll notice that `STUSPS` contains state abbreviations.

We can make a simple list of them then locate them and save those rows as a new, smaller GeoDataFrame:

In [ ]:
example_state_abbreviations = ["CA", "NV", "OR", "AZ"]

california_states = states.loc[
    states["STUSPS"].isin(example_state_abbreviations)
]

california_states[["NAME", "STUSPS", "geometry"]]

And then GeoPandas easy-peasy `.plot` allows us to view them:

In [ ]:
ax = california_states.plot(
    facecolor="lightgreen",
    edgecolor="darkgreen",
    linewidth=0.8,
    figsize=(8, 6)
)

ax.set_title("Selected State Boundaries")
ax.set_axis_off()

But what if we wanted to make the states different colors?

We can do that easily by first choosing colors and adding them to our GeoDataFrame as a new column called "color":

In [ ]:
# dictionary of colors
state_colors = {
    "CA": "gold",
    "NV": "darkred",
    "OR": "darkgreen",
    "AZ": "salmon",
}

# map method to map the colors to the abbrevs
california_states["color"] = california_states["STUSPS"].map(state_colors)

Then when we assign the `color` option in the plot, we just point to the color column values, like this:

In [ ]:
ax = california_states.plot(
    color=california_states["color"],
    edgecolor="black",
    linewidth=0.8,
    figsize=(8, 6)
)

ax.set_title("Selected State Boundaries")
ax.set_axis_off()

## Create a Contiguous U.S. Base Map

But what if we wanted just a map of the lower 48 states?

That's easy enough. We could make a list of all the abbreviations of all the 48 contiguous states, but how bout we work in the other direction: we'll make a list of the non-contiguous states and territories and filter them out of the GeoDataFrame.

This is easy with the tilde `~`: the symbol in Python for "not this" or "opposite of this":

In [ ]:
non_contiguous = ["AK", "HI", "PR", "GU", "VI", "MP", "AS"]

states_48 = states.loc[
    # the ~ indicates "not these"
    ~states["STUSPS"].isin(non_contiguous)
].copy()

states_48.shape

In [ ]:
ax = states_48.plot(
    facecolor="white",
    edgecolor="black",
    linewidth=0.5,
    figsize=(14, 8)
)

ax.set_title("Contiguous U.S. State Boundaries")
ax.set_axis_off()

# Mapping Chronicling America Newspapers

Now let's work with some coordinate data. In our `data` folder, you'll find a csv called `chron_am_papers.csv`. It contains data about digitized newspapers from the [Chronicling America archive](https://www.loc.gov/collections/chronicling-america/about-this-collection/). I downloaded and preprocessed it a bit, making it ready for this tutorial.

This file is an ordinary CSV, so we start with pandas. Then we will turn it into a GeoDataFrame by creating a Shapely `Point` from each row's `Geo Location` value.

Library roadmap for this part: `pandas` reads and cleans the CSV, `Shapely` turns longitude/latitude pairs into point geometries, `GeoPandas` stores those points in a spatial table, and then `GeoPandas` plus `matplotlib` create static maps.

In [ ]:
papers = pd.read_csv("../data/chron_am_papers.csv")

papers.shape

In [ ]:
papers.head()

In [ ]:
papers.columns

## Preprocessing the Geolocations

Before mapping, we need to turn the `Geo Location` text column into clean numeric coordinates and then into Shapely point geometries. We'll do that in a few small steps: check for missing locations, split the coordinate text into latitude and longitude, drop rows without usable coordinates, and create points for the remaining rows.

Let's start by splitting the `Geo Location` values into latitude and longitude. If you look at just one example, you'll see the `Geo Location` data contains both coordinates at once:

In [ ]:
papers["Geo Location"][0]

Also, what data type is contained in the `Geo Location` column?

In [ ]:
papers["Geo Location"].dtypes

Looks like we have strings and some nan values (empty or no data rows). We'll have to convert these data to be compatible with our mapping libraries. We can start by splitting `Geo Location` using the .split() method, like this:

In [ ]:
papers[["latitude", "longitude"]] = papers["Geo Location"].str.split(",", expand=True)
papers.head()

That worked! Let's check the data types in these new lat/long columns:

In [ ]:
papers[["latitude", "longitude"]].dtypes

They are strings. Strings aren't compatible with geospatial mapping––we need to convert them to floats. We can do that with the `.apply()` method in Pandas and the `pd.to_numeric` function, like this:

In [ ]:
papers["latitude"] = papers["latitude"].apply(pd.to_numeric, errors="coerce")

papers["longitude"] = papers["longitude"].apply(pd.to_numeric, errors="coerce")

papers[["Geo Location", "latitude", "longitude"]].dtypes

`pd.to_numeric()` turned the text values into numbers. The argument `errors="coerce"` means that anything Python cannot convert becomes a missing value, which pandas represents as `NaN`.

This conversion matters because `Shapely` needs numeric longitude and latitude values before it can create map points. Let's check to see if we have any `NaN` values:

In [ ]:
papers[["latitude", "longitude"]].isna().sum()

In [ ]:
missing_coordinates = papers[
    papers[["latitude", "longitude"]].isna().any(axis=1)
]

missing_coordinates

Looks like there's one `NaN` row. We can identify it and review it like this:

In [ ]:
missing_coordinate = papers[papers[["latitude", "longitude"]].isna().any(axis=1)]

missing_coordinate

If we wanted, we could look up the coordinates for this newspaper and enter them into the data, but let's just skip it for now. We'll just drop this row from our data, like this:

In [ ]:
papers_geo = papers.dropna(subset=["latitude", "longitude"])

print("Original rows:", len(papers))
print("Rows with coordinates:", len(papers_geo))

And now we're ready to convert this data into mappable points. To do that, we'll convert it into Shapely points. Remember: Shapely expects `Point(longitude, latitude)`. These Shapely points will make the data compatible with GeoPandas.

Let's loop over our data and convert it to Shapely points. To do that, we'll create an empty list shell (accumulator) and a for loop. In the for-loop, we'll use `.iterrows()` and the Shapely `Point()` function, like this:

In [ ]:
geometry = []

for index, row in papers_geo.iterrows():
    point = Point(row["longitude"], row["latitude"])
    geometry.append(point)

geometry[:5]

## Create a GeoDataFrame

Now we'll combine our Pandas dataframe with the point geometries into a GeoDataFrame, and Viola! We're ready to map. 

Notice again how we're being sure to assign the crs. And we're adding the `geometry=` column––the defining feature of GeoDataFrames––as the list of Shapely points:

In [ ]:
papers_gdf = gpd.GeoDataFrame(
    papers_geo,
    geometry=geometry,
    crs="EPSG:4326"
)

papers_gdf.head()

## Make a First Static Newspaper Map

With our new GeoDataFrame, we're now ready to map it! We can do that easily again with the `.plot()` function. To map coordinates, we just have to use some different options (`markersize=` and `alpha=`). Let's try it out:

In [ ]:
ax = papers_gdf.plot(
    color="darkgreen",
    markersize=5,
    alpha=1,
    figsize=(10, 6)
)

ax.set_title("Chronicling America Newspapers with Geolocations")
ax.set_axis_off()

Hmmm, okay, that's definitely a map. But what's going on here?

It doesn't have boundaries, and it's got so many coordinates outside the contiguous US... Maybe we should try to narrow things down? And we should add a boundary layer so we can show the state boundaries, like we did before.

Let's start by narrowing to only the newspapers located within the contiguous US. There are a few ways to do this, but let's think geospatially––let's look up the rough boundaries of the contiguous US in terms of lat/long.

A quick Google search tells me the contiguous US goes from about -125 to -66 (longitude) and about 24 to 50 (latitude). With those numbers identified, we can use `.between()`––a Pandas method that will let us subset by any numbers between those values.

Let's do it with .loc and .between(), like this:

In [ ]:
papers_48 = papers_gdf.loc[
    papers_gdf["longitude"].between(-125, -66)
    & papers_gdf["latitude"].between(24, 50)
].copy()

print(f"Locations before contiguous subsetting: {len(papers_gdf)}")
print(f"Locations after contiguous subsetting: {len(papers_48)}")

Very good. Now if we map our papers_48 GeoDataFrame, it should be limited to just the contiguous US:

In [ ]:
ax = papers_48.plot(
    color="darkgreen",
    markersize=5,
    alpha=1,
    figsize=(10, 6)
)

ax.set_title("Chronicling America Newspapers with Geolocations")
ax.set_axis_off()

To add state boundaries, we need to first plot the state_48 layer. Then, in GeoPandas, we can just add the plotted points as a new layer atop the state_48 layer, like this:

In [ ]:
ax = states_48.plot(
    facecolor="white",
    edgecolor="black",
    linewidth=0.5,
    figsize=(14, 8)
)

papers_48.plot(
    ax=ax,
    color="darkgreen",
    markersize=5,
    alpha=0.35
)

ax.set_title("Chronicling America Newspapers with Geolocations")
ax.set_axis_off()

## Why Some Points Overlap

That works! But there's another thing we need to account for. Our map has lots of overlapping points. This is because many newspapers share the same city. If five newspapers are all listed in the same city, they may have the exact same latitude and longitude. On a map, those points sit directly on top of one another and become indistinguishable from one another.

Let's count the most repeated coordinates. This will reveal which papers are overlapping:

Notice the .groupby() method, size to get the count of the groupings, and so forth. It's the same process as in previous tutorials for getting duplicate entry counts.

In [ ]:
location_counts = (
    papers_gdf
    .groupby(["latitude", "longitude"])
    .size()
    .reset_index(name="paper_count")
    .sort_values("paper_count", ascending=False)
)

location_counts.head(10)

## Jitter Duplicate Locations

Overlapping coordinate points is a common problem among geospatial projects. To deal with it, geospatial programmers have come up with a method called __jittering__. Jittering means adding a tiny random shift to point locations, thus making them mappable. This helps us see overlapping records on the map.

Let's start by making copies of our `latitude` and 'longitude` columns:

In [ ]:
papers_gdf["display_latitude"] = papers_gdf["latitude"]
papers_gdf["display_longitude"] = papers_gdf["longitude"]

Now we'll jitter these new column lat/longs. First we'll create a NumPy random number generator. The seed=42 (secret of universe) makes the random numbers reproducible, so the jitter will look the same each time the notebook is run. The jitter_degrees sets the maximum size of the random shift. Here, each point can move by up to 0.1 degrees in either direction.

In [ ]:
rng = np.random.default_rng(seed=42)
jitter_degrees = 0.1

Then we'll set one random latitude shift for every row in papers_gdf. Each value will be somewhere between -0.1 and 0.1.

In [ ]:
lat_jitter = rng.uniform(
    low=-jitter_degrees,
    high=jitter_degrees,
    size=len(papers_gdf)
)

And the same for longitudes:

In [ ]:
lon_jitter = rng.uniform(
    low=-jitter_degrees,
    high=jitter_degrees,
    size=len(papers_gdf)
)

And now we've got our random numbers. Just briefly, here's what we created:

In [ ]:
print(lat_jitter[0:4])
print(lon_jitter[0:4])

And finally, we'll add these little numbers to our display lat/longs, thereby making them slightly random and off from one another, so they're better for mapping:

In [ ]:
papers_gdf["display_latitude"] = papers_gdf["display_latitude"] + lat_jitter
papers_gdf["display_longitude"] = papers_gdf["display_longitude"] + lon_jitter

Now we'll need to rebuild the geometry column so that it uses the display coordinates. This is just repeating the Shapely object step, and making our GeoDataFrame geometry column the jittered version:

In [ ]:
display_geometry = [
    Point(longitude, latitude)
    for longitude, latitude in zip(
        papers_gdf["display_longitude"],
        papers_gdf["display_latitude"]
    )
]

papers_gdf = gpd.GeoDataFrame(
    papers_gdf,
    geometry=display_geometry,
    crs="EPSG:4326"
)

Let's look at a few examples of the differences now:

In [ ]:
papers_gdf[["latitude", "longitude", "display_latitude", "display_longitude",]].head()

Let's compare these jittered points to our previous map now. Do you notice a difference? Are there more points visible?

In [ ]:
papers_display_48 = papers_gdf.loc[
    papers_gdf["display_longitude"].between(-125, -66)
    & papers_gdf["display_latitude"].between(24, 50)
].copy()

ax = states_48.plot(
    facecolor="white",
    edgecolor="black",
    linewidth=0.5,
    figsize=(14, 8)
)

papers_display_48.plot(
    ax=ax,
    color="darkgreen",
    markersize=5,
    alpha=0.35
)

ax.set_title("Jittered Newspaper Locations")
ax.set_axis_off()

# Scale Points by Number of Issues

I think I see a difference, but perhaps we can make this map a little more informative. If we go back to the other columns, you'll notice there's a `Number of Issues` column that gives us the number of newspaper issues per newspaper. This is a variable we can encode visually. That is, larger values can be shown with larger points.

Let's look at this data variable using the `.describe()` method:

In [ ]:
# make sure we read as a number
papers_gdf["Number of Issues"] = pd.to_numeric(papers_gdf["Number of Issues"], errors="coerce")

papers_gdf["Number of Issues"].describe()

It looks like the issue counts range quite a bit. A few papers have a very large number of issues while others have only one. What would it look like if we scaled the points on the map to the number of newspaper issues per newspaper?

Let's try it out. We actually don't need to change much of anything to do it, too. All we need to edit from our previous map is the `markersize=` option. We'll set it to the `Number of Issues` column values, like this:

In [ ]:
papers_display_48 = papers_gdf.loc[
    papers_gdf["display_longitude"].between(-125, -66)
    & papers_gdf["display_latitude"].between(24, 50)
].copy()

ax = states_48.plot(
    facecolor="white",
    edgecolor="black",
    linewidth=0.5,
    figsize=(14, 8)
)

papers_display_48.plot(
    ax=ax,
    color="darkgreen",
    markersize=papers_display_48["Number of Issues"], # all that needs to change here
    alpha=0.35
)

ax.set_title("Jittered Newspaper Locations")
ax.set_axis_off()

Ummm, that's something, but not quite right. The range of numbers of issues is just too much! Some newspapers completely dominate the map.

So, we can't use the raw numbers of issues per newspaper. Instead, let's try a __square-root transformation__. This will keep the differences visible but less extreme. This is just limiting the range to the square roots of the min and max radius.

To do that, we'll use `NumPy`'s `sqrt` method and assign the results to a series sqrt_issues:

In [ ]:
sqrt_issues = np.sqrt(papers_gdf["Number of Issues"])
print(type(sqrt_issues))
print(sqrt_issues.head())

Then we'll do some standardizing of these square roots, making their range smaller. This is admittedly a little complicated... Math! Yuck! But let's break it down:

First we'll create min_radius and max_radius, setting the smallest and largest possible points. Then we'll take the square-root issue counts, convert them to a 0-to-1 scale, stretch that scale to fit between the smallest and largest marker sizes we want, and save the result as scaled_radius.

Then we need to convert scaled_radius to area so it works nicely with our libraries. In our case, we'll square the radius-like values we've created with `**` then divide them by 2.

In [ ]:
min_radius = 3
max_radius = 40

scaled_radius = (
    min_radius
    + (sqrt_issues - sqrt_issues.min())
    / (sqrt_issues.max() - sqrt_issues.min())
    * (max_radius - min_radius)
)

papers_gdf["issue_marker_size"] = scaled_radius ** 2 / 2

Let's see what that's done for us:

In [ ]:
papers_gdf.sort_values("Number of Issues", ascending=False).head()

All right, now let's map it using the issue_marker_size column as our `markersize=` values:

In [ ]:
papers_display_48 = papers_gdf.loc[
    papers_gdf["display_longitude"].between(-125, -66)
    & papers_gdf["display_latitude"].between(24, 50)
].copy()

ax = states_48.plot(
    facecolor="white",
    edgecolor="black",
    linewidth=0.5,
    figsize=(14, 8)
)

papers_display_48.plot(
    ax=ax,
    color="darkgreen",
    markersize=papers_display_48["issue_marker_size"],
    alpha=0.35
)

ax.set_title("Newspaper Points Scaled by Number of Issues")
ax.set_axis_off()

# Interactive Mapping with Folium

All right, we've had our fun with GeoPandas, but it's not the only mapping library out there. GeoPandas is great for static maps. But if we wanted to make an interactive map, something like Folium is better.

Folium builds maps as HTML documents. Behind the scenes, it uses a JavaScript mapping library called Leaflet. When we create a Folium map in Python, Folium writes the HTML and JavaScript needed to display that map in a notebook or save it as an `.html` file. This may sound complicated but don't fear: all you need to do is work in Python, and Folium does the converting for you.

Folium also includes basemap tiles, which are the background map images. Because those tiles already show political boundaries, roads, labels, and other reference information, we don't need to add things like state boundaries here.

In this Folium section, we will create:

- a base map
- one circle marker per geolocated newspaper
- hover text with the newspaper name and location
- simple popup text with a few useful details

Library note: from this point forward, `folium` handles the interactive map. We will still use values from our GeoDataFrame, but Folium markers use numeric latitude and longitude values directly rather than the Shapely geometry column.

Let's start by calling a simple map using Folium. Notice the `location=` gives it a centralized point, the `zoom_start=` tells it how far the zoom begins, and the `tiles=` value tells it what map background we're working with.

For more options, check out the [Folium documentation here.](https://python-visualization.github.io/folium/latest/user_guide.html)

In [ ]:
newspaper_map = folium.Map(
    location=[39.5, -98.35],
    zoom_start=4,
    tiles="CartoDB positron"
)

newspaper_map

## Choose Rows for the Interactive Map

Before adding markers, we'll use the same latitude/longitude limits from the static maps so the interactive map focuses on the contiguous United States.

Let's borrow our code for that subsetting:

In [ ]:
papers_for_map = papers_gdf.loc[
    papers_gdf["display_longitude"].between(-125, -66)
    & papers_gdf["display_latitude"].between(24, 50)
].copy()

papers_for_map.shape

## Add Newspaper Markers

Now we'll loop through the newspaper rows and add one circle marker for each row.

Each marker needs a `location`, which Folium expects in `[latitude, longitude]` order. Notice that this is the opposite order from Shapely's `Point(longitude, latitude)`.

We'll also add a simple tooltip and popup inside the same loop. These are little functionalities of Folium maps. The tooltip appears when someone hovers over a marker. The popup appears when someone clicks a marker. In the loop, we're building these components to feed into Folium. Then Folium converts them into the proper HTML map document.

In [ ]:
for index, row in papers_for_map.iterrows():
    # notice the html here
    popup_text = (
        f"{row['Newspapers']}<br>"
        f"Number of issues: {row['Number of Issues']}"
    )

    tooltip_text = f"{row['Newspapers']} ({row['City']}, {row['State']})"

    folium.CircleMarker(
        location=[row["display_latitude"], row["display_longitude"]],
        radius=4,
        color="darkgreen",
        weight=1,
        fill=True,
        fill_color="darkgreen",
        fill_opacity=0.45,
        tooltip=tooltip_text,
        popup=folium.Popup(popup_text, max_width=300),
    ).add_to(newspaper_map)

## Display the Interactive Map

Now we display the Folium map. In Jupyter, the map appears as an embedded HTML object. You can pan, zoom, hover over markers, and click markers to open their popups. Simple as that!

In [ ]:
newspaper_map

## Optional: Save the Folium Map

Folium maps are HTML documents. If you save the map, Folium writes a standalone `.html` file that can be opened in a browser. The file contains the map structure, marker data, popup text, and JavaScript needed to display the interactive map.

To save the map, uncomment and run the line below.

In [ ]:
# newspaper_map.save("chron_am_newspaper_map.html")